# Reproducible Data Workflow for Video Game Sales, Ratings, and Genre Trends

**Student:** Robert Mayfield  
**Project:** AI Programming Foundations - Udacity AI MBA Capstone  
**Dataset:** Video Game Sales, Ratings, and Genre Data

## 1. Project Overview

This project focuses on building a clean, reusable, and reproducible data workflow using Python, Pandas, NumPy, Matplotlib, and Seaborn. The `Video Game Sales and Ratings` dataset contains video game market information such as game titles, platforms, genres, publishers, sales figures, ratings, and review scores. The goal is to clean and explore the dataset, create meaningful visualizations, and summarize insights that help establish industry context for a future AI-assisted game development system called AI Game Director Studio.

### Core Question

What patterns in video game genre, platform, sales, and ratings data can inform the design focus of an AI assisted game development tool?

## 2. Setup

The following imports support data loading and initial inspection. Additional libraries will be added in later sections as needed.

In [12]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import zipfile
import pandas as pd

from src.inspection import inspect_dataframe
from src.cleaning import clean_column_names, clean_score_columns, handle_missing_values

## 3. Data Ingestion

The dataset is stored as a zip archive at `data/raw/video-game-sales-ratings-data.zip` and contains a single CSV file, `Video_Games.csv`. The data is loaded directly from the archive without extracting to disk, keeping the raw data directory clean and the workflow self contained.

In [13]:
ZIP_PATH = '../data/raw/video-game-sales-ratings-data.zip'
CSV_NAME = 'Video_Games.csv'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open(CSV_NAME) as f:
        df = pd.read_csv(f, index_col=0)

df.head()

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
index,,,,,,,,,,,,,,,,
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Initial Dataset Inspection

Before cleaning or analysis, we run a full inspection of the dataset to understand its structure, identify missing values, spot duplicates, and get a sense of the distributions across numeric and categorical columns. The `inspect_dataframe` function from `src/inspection.py` packages these steps into a single reusable call.

In [14]:
inspect_dataframe(df)

SHAPE AND STRUCTURE
------------------------------------------------------------
Rows:    16,928
Columns: 16

COLUMN NAMES AND DATA TYPES
------------------------------------------------------------
Name                object
Platform            object
Year_of_Release    float64
Genre               object
Publisher           object
NA_Sales           float64
EU_Sales           float64
JP_Sales           float64
Other_Sales        float64
Global_Sales       float64
Critic_Score       float64
Critic_Count       float64
User_Score          object
User_Count         float64
Developer           object
Rating              object

MEMORY USAGE
------------------------------------------------------------
Total: 8.19 MB

MISSING VALUES
------------------------------------------------------------
                 Null Count  Null %
User_Count             9210   54.41
Critic_Count           8668   51.21
Critic_Score           8668   51.21
Rating                 6836   40.38
User_Score            

### Inspection Notes

**Shape and Structure:** The dataset contains 16,928 rows and 16 columns. Columns cover game titles, platforms, release years, genres, publishers, four regional sales figures, global sales, critic and user review scores, developer names, and ESRB ratings. Total memory footprint is 8.19 MB.

**Data Types:** All sales and score columns load as float64 except `User_Score`, which loads as object. The categorical describe output reveals why: the most frequent value in `User_Score` is `tbd`, a placeholder used when a user score is not yet available. This string presence prevents pandas from inferring a numeric type. `Year_of_Release` also loads as float64 rather than integer, likely due to missing values in that column. Both will be addressed during cleaning.

**Missing Values:** The dataset has substantial missingness concentrated in review related columns. `User_Count` is missing for 9,210 records (54.41%), `Critic_Count` and `Critic_Score` are each missing for 8,668 records (51.21%), and `Rating`, `User_Score`, and `Developer` are missing for roughly 40% of records each. This pattern is consistent across the dataset: older and lower profile titles were frequently not reviewed by critics or users. `Year_of_Release` is missing for 273 records (1.61%), `Publisher` for 55 (0.32%), and `Name` and `Genre` for just 2 records each (0.01%). The review column gaps are significant enough that any analysis involving scores will be working with roughly half the dataset.

**Duplicate Rows:** 209 duplicate rows are present and will be removed during cleaning.

**Numeric Distributions:** Sales figures are heavily right skewed across all regions. The median global sales figure is 0.17 million units while the maximum is 82.53 million, and the 75th percentile sits at just 0.49 million. The vast majority of titles sell modestly, with a small number of blockbusters driving the upper range. `Critic_Score` ranges from 13 to 98 with a mean of 69, suggesting most reviewed titles cluster in the average to good range. `User_Count` has an extreme distribution with a mean of 163 but a maximum of 10,665 and a large standard deviation, indicating that a handful of titles attract enormous user review volume while most receive very few. Release years span 1980 to 2020 with a mean of 2006.5, reflecting that the bulk of the dataset covers the mid-2000s peak of the disc-based console era.

**Categorical Summaries:** There are 31 unique platforms in the dataset, with PS2 being the most represented at 2,188 titles. Genre has only 12 unique values, with Action being the most common at 3,410 titles. Publisher has 581 unique values, with Electronic Arts appearing most frequently at 1,389 titles. Developer has 1,696 unique values. Rating has 8 unique values with E (Everyone) being the most common at 4,043 titles, followed by other ESRB categories.

**Sample Data:** The first rows confirm the dataset is sorted by global sales descending, with Nintendo titles dominating the top. The last rows show titles clustered around 0.6 million in global sales, and several are missing review data entirely, consistent with the missing value patterns described above.

### Extended Missingness Analysis

The high missingness in review columns warrants a closer look before making cleaning decisions. The following cells examine whether gaps are random or concentrated in specific years, platforms, or genres, and whether the review columns tend to go missing together.

#### Critic Score Missingness by Release Year

In [15]:
year_df = df.dropna(subset=['Year_of_Release']).copy()
year_df['Year_of_Release'] = year_df['Year_of_Release'].astype(int)

missing_by_year = (
    year_df.groupby('Year_of_Release')['Critic_Score']
    .apply(lambda x: x.isnull().mean() * 100)
    .round(1)
    .reset_index()
    .rename(columns={'Critic_Score': 'Critic_Score_Missing_%'})
)

print(missing_by_year.to_string(index=False))

 Year_of_Release  Critic_Score_Missing_%
            1980                   100.0
            1981                   100.0
            1982                   100.0
            1983                   100.0
            1984                   100.0
            1985                    92.9
            1986                   100.0
            1987                   100.0
            1988                    93.3
            1989                   100.0
            1990                   100.0
            1991                   100.0
            1992                    97.7
            1993                   100.0
            1994                    99.2
            1995                   100.0
            1996                    97.0
            1997                    94.2
            1998                    92.2
            1999                    88.6
            2000                    59.0
            2001                    32.0
            2002                    24.1
            2003

#### Critic Score Missingness by Platform

In [16]:
missing_by_platform = (
    df.groupby('Platform')['Critic_Score']
    .apply(lambda x: x.isnull().mean() * 100)
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'Critic_Score': 'Critic_Score_Missing_%'})
)

print(missing_by_platform.to_string(index=False))

Platform  Critic_Score_Missing_%
    2600                   100.0
     3DO                   100.0
      GB                   100.0
     NES                   100.0
     N64                   100.0
      GG                   100.0
     GEN                   100.0
    PCFX                   100.0
      NG                   100.0
     SCD                   100.0
      WS                   100.0
    SNES                   100.0
    TG16                   100.0
     SAT                   100.0
      PS                    83.2
      DC                    73.1
     PSV                    72.2
     3DS                    67.5
      DS                    66.7
     PSP                    61.8
     Wii                    55.6
     GBA                    46.4
     PS2                    39.8
    WiiU                    38.5
     PS3                    37.9
     PS4                    36.1
    XOne                    31.6
    X360                    27.2
      PC                    26.8
      GC  

#### Critic Score Missingness by Genre

In [17]:
missing_by_genre = (
    df.dropna(subset=['Genre']).groupby('Genre')['Critic_Score']
    .apply(lambda x: x.isnull().mean() * 100)
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'Critic_Score': 'Critic_Score_Missing_%'})
)

print(missing_by_genre.to_string(index=False))

       Genre  Critic_Score_Missing_%
   Adventure                    75.3
        Misc                    69.9
      Puzzle                    61.5
  Simulation                    59.7
    Strategy                    55.9
    Fighting                    51.6
Role-Playing                    50.8
      Sports                    48.9
    Platform                    44.1
      Action                    43.9
      Racing                    40.5
     Shooter                    28.4


#### Records Missing Year of Release

In [18]:
missing_year = df[df['Year_of_Release'].isnull()]
print(f"Records missing Year_of_Release: {len(missing_year)}\n")
print(missing_year[['Name', 'Platform', 'Genre', 'Publisher', 'Global_Sales']].to_string())

Records missing Year_of_Release: 273

                                                                     Name Platform         Genre                               Publisher  Global_Sales
index                                                                                                                                                 
183                                                       Madden NFL 2004      PS2        Sports                         Electronic Arts          5.23
377                                                      FIFA Soccer 2004      PS2        Sports                         Electronic Arts          3.49
456                                            LEGO Batman: The Videogame      Wii        Action  Warner Bros. Interactive Entertainment          3.06
475                                            wwe Smackdown vs. Raw 2006      PS2      Fighting                                     NaN          3.00
609                                                     

#### Review Column Co-occurrence of Missingness

In [19]:
review_cols = ['Critic_Score', 'Critic_Count', 'User_Score', 'User_Count', 'Developer', 'Rating']

# How many review columns are missing per record
missing_per_record = df[review_cols].isnull().sum(axis=1).value_counts().sort_index()
print("Number of review columns missing per record:")
for n_missing, count in missing_per_record.items():
    pct = count / len(df) * 100
    print(f"  {n_missing} columns missing: {count:,} records ({pct:.1f}%)")

print()

# Do Critic_Score and User_Score tend to go missing together?
both_missing = df[df['Critic_Score'].isnull() & df['User_Score'].isnull()].shape[0]
critic_only  = df[df['Critic_Score'].isnull() & df['User_Score'].notna()].shape[0]
user_only    = df[df['Critic_Score'].notna()  & df['User_Score'].isnull()].shape[0]

print("Critic_Score and User_Score co-occurrence:")
print(f"  Both missing:          {both_missing:,}")
print(f"  Only Critic_Score missing: {critic_only:,}")
print(f"  Only User_Score missing:   {user_only:,}")

Number of review columns missing per record:
  0 columns missing: 7,063 records (41.7%)
  1 columns missing: 1,151 records (6.8%)
  2 columns missing: 603 records (3.6%)
  3 columns missing: 1,326 records (7.8%)
  4 columns missing: 76 records (0.4%)
  5 columns missing: 54 records (0.3%)
  6 columns missing: 6,655 records (39.3%)

Critic_Score and User_Score co-occurrence:
  Both missing:          6,731
  Only Critic_Score missing: 1,937
  Only User_Score missing:   38


### Extended Missingness Findings and Cleaning Decisions

**Missingness by release year** reveals a sharp and systematic pattern. Titles released before 2000 are almost entirely missing critic scores, with most years in the 1980s and 1990s at or near 100% missingness. The coverage improves dramatically around 2000 to 2001, dropping to 32% missing, which aligns with the rise of dedicated games journalism and online review aggregation. Missingness then climbs again in the later years of the dataset, reaching 100% for 2017 and 2020, which likely reflects incomplete data collection rather than a lack of reviews. The missingness is not random. It is a direct function of when structured review data became consistently available.

**Missingness by platform** reinforces this conclusion. Every retro and legacy platform (NES, SNES, N64, GB, GEN, 2600, and others) shows 100% missingness. Older platforms that straddle the pre and post review era such as PS1 (83.2%) and DC (73.1%) show high but not complete gaps. Portable platforms (PSV 72.2%, 3DS 67.5%, DS 66.7%, PSP 61.8%) show elevated missingness even in the modern era, suggesting that handheld titles received less consistent critical coverage than their console counterparts. The most modern, high profile platforms (XB 11.9%, GC 19.5%, PC 26.8%, X360 27.2%) have the lowest missingness.

**Missingness by genre** shows a wider spread than the year and platform patterns. Adventure (75.3%) and Misc (69.9%) genres are most affected, while Shooter (28.4%) has the lowest missingness. This likely reflects that shooter and action titles attracted more consistent critical attention while niche genres such as adventure and simulation received less formal review coverage.

**Co-occurrence of missingness** is the most important finding for the cleaning decision. The data is strongly bimodal: 41.7% of records have none of the six review columns missing, and 39.3% have all six missing. Only 18.9% of records fall somewhere in between. This means the dataset contains two largely distinct populations, well reviewed games with complete data and unreviewed games with no review data at all. Critic_Score and User_Score go missing almost entirely together: 6,731 records are missing both, while only 38 records have a User_Score but no Critic_Score.

**Records missing Year_of_Release** include recognizable titles such as Madden NFL 2004, FIFA Soccer 2004, and LEGO Batman, indicating these are real games with a data collection gap rather than erroneous entries. Since year is required for any time based analysis and only 273 records are affected (1.61%), dropping these rows is a reasonable and low impact decision.

---

**Cleaning Decisions**

Based on this analysis, the following decisions will guide the cleaning functions:

| Column(s) | Decision | Reasoning |
|---|---|---|
| Duplicate rows (209) | Drop | Exact duplicates add no information |
| `Name`, `Genre` (2 missing each) | Drop rows | Cannot analyze a game without a title or genre |
| `Publisher` (55 missing) | Drop rows | Small count, publisher needed for meaningful grouping |
| `Year_of_Release` (273 missing) | Drop rows | Required for time analysis, small percentage |
| `Critic_Score`, `Critic_Count`, `User_Score`, `User_Count`, `Developer`, `Rating` | Keep NaN | Missingness is systematic and era driven, not a data error. Dropping would eliminate nearly all pre 2000 titles and bias the dataset toward the modern era. Score based analyses will drop nulls at the point of use. |
| `User_Score` `tbd` values | Replace with NaN, convert to float | `tbd` is a placeholder, not a score |

## 5. Data Cleaning Functions

Three cleaning functions are defined in `src/cleaning.py` and applied in sequence below. Each function returns a copy of the DataFrame, preserving the raw data unchanged. The cleaning decisions applied here are documented and justified in the extended missingness analysis in Section 4.

- `clean_column_names` standardizes all column names to lowercase.
- `clean_score_columns` replaces `tbd` placeholder values in `user_score` with NaN and converts the column to float.
- `handle_missing_values` drops duplicate rows and rows missing values in essential columns, then converts `year_of_release` to integer.

In [20]:
rows_before = len(df)

df_clean = clean_column_names(df)
df_clean = clean_score_columns(df_clean)
df_clean = handle_missing_values(df_clean)

rows_after = len(df_clean)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning:  {rows_after:,}")
print(f"Rows removed:         {rows_before - rows_after:,}")
print()
print(f"Columns: {list(df_clean.columns)}")
print(f"user_score dtype: {df_clean['user_score'].dtype}")
print(f"year_of_release dtype: {df_clean['year_of_release'].dtype}")

Rows before cleaning: 16,928
Rows after cleaning:  16,416
Rows removed:         512

Columns: ['name', 'platform', 'year_of_release', 'genre', 'publisher', 'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'global_sales', 'critic_score', 'critic_count', 'user_score', 'user_count', 'developer', 'rating']
user_score dtype: float64
year_of_release dtype: int64


In [21]:
df_clean.head()

,name,platform,year_of_release,genre,publisher,na_sales,eu_sales,jp_sales,other_sales,global_sales,critic_score,critic_count,user_score,user_count,developer,rating
0,Wii Sports,Wii,2006,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8.0,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8.0,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN


#### Cleaning Verification

In [ ]:
checks = {
    "No duplicate rows":                df_clean.duplicated().sum() == 0,
    "No nulls in name":                 df_clean['name'].isnull().sum() == 0,
    "No nulls in genre":                df_clean['genre'].isnull().sum() == 0,
    "No nulls in publisher":            df_clean['publisher'].isnull().sum() == 0,
    "No nulls in year_of_release":      df_clean['year_of_release'].isnull().sum() == 0,
    "user_score is float64":            df_clean['user_score'].dtype == 'float64',
    "year_of_release is int64":         df_clean['year_of_release'].dtype == 'int64',
    "No 'tbd' values in user_score":    'tbd' not in df_clean['user_score'].astype(str).values,
    "All column names are lowercase":   all(c == c.lower() for c in df_clean.columns),
}

all_passed = True
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_passed = False
    print(f"  [{status}] {check}")

print()
print("All checks passed." if all_passed else "One or more checks failed.")

#### Save Cleaned Dataset

In [ ]:
PROCESSED_PATH = '../data/processed/video_games_clean.csv'

df_clean.to_csv(PROCESSED_PATH, index=False)

print(f"Cleaned dataset saved to {PROCESSED_PATH}")
print(f"Shape: {df_clean.shape}")

### Cleaning Notes

**Rows removed:** The pipeline removed 512 rows total, reducing the dataset from 16,928 to 16,416. The 512 reflects the combined effect of dropping 209 duplicate rows and dropping rows missing essential field values across name, genre, publisher, and year_of_release, with some rows satisfying more than one removal condition.

**Column names:** All 16 column names are now lowercase, making them consistent and easier to reference throughout the rest of the notebook.

**user_score:** The column is now float64. The `tbd` placeholder values have been replaced with NaN and the column successfully converted from object to numeric.

**year_of_release:** Now int64. The conversion is safe because all rows with a missing year were dropped before the cast, eliminating the fractional representation that float requires for nullable integers.

**Review columns retained:** `critic_score`, `critic_count`, `user_score`, `user_count`, `developer`, and `rating` retain their NaN values. As established in the missingness analysis, these gaps are systematic and era driven. Analyses that require score data will filter to non-null rows at the point of use rather than discarding this data globally.

## 6. Exploratory Analysis Function

## 7. Visualizations

## 8. Summary and Interpretation

## 9. Responsible Data Handling and Bias Awareness

## 10. Future Integration with AI Game Director Studio